In [3]:
import requests
import pandas as pd
import time

# --- KONFIGURACJA ---
api_token = 'token'  # Twój klucz

# Lista symboli, których używasz w projekcie (format Finnhub/Twój)
SYMBOLS_LIST = [
    "AAPL", "AMZN", "NVDA", "INTC", "AMD", "BRK.B",
    "BINANCE:BTCUSDT", "BINANCE:ETHUSDT"
]

# Mapa tłumacząca Twoje nazwy na format EODHD
# EODHD wymaga: .US dla akcji USA, .CC dla krypto, myślnika zamiast kropki dla BRK
SYMBOL_MAP = {
    "AAPL": "AAPL.US",
    "AMZN": "AMZN.US",
    "NVDA": "NVDA.US",
    "INTC": "INTC.US",
    "AMD": "AMD.US",
    "BRK.B": "BRK-B.US",       # Uwaga na myślnik!
    "BINANCE:BTCUSDT": "BTC-USD.CC",
    "BINANCE:ETHUSDT": "ETH-USD.CC"
}

all_data = []  # Tu będziemy zbierać wszystkie wiersze

print(f"--- Rozpoczynam pobieranie dla {len(SYMBOLS_LIST)} symboli ---")

for user_symbol in SYMBOLS_LIST:
    # Pobieramy odpowiednik dla EODHD
    eod_symbol = SYMBOL_MAP.get(user_symbol)

    if not eod_symbol:
        print(f"Pominięto: brak mapowania dla {user_symbol}")
        continue

    # Budowanie URL (pobieramy rok 2025)
    url = f'https://eodhd.com/api/eod/{eod_symbol}?from=2025-01-01&to=2025-12-30&period=d&api_token={api_token}&fmt=json'

    try:
        print(f"Pobieranie: {user_symbol} ({eod_symbol})...")
        response = requests.get(url)

        if response.status_code == 200:
            data = response.json()

            # WAŻNE: Jeśli API zwróci pustą listę lub błąd w JSONie
            if isinstance(data, list) and len(data) > 0:
                # Dodajemy kolumnę 'symbol' do każdego wiersza w pobranych danych
                for row in data:
                    row['original_symbol'] = user_symbol  # Symbol taki jak w Twojej Kafce
                    row['eod_symbol'] = eod_symbol        # Symbol techniczny EODHD

                # Doklejamy do głównej listy
                all_data.extend(data)
                print(f"   -> OK. Pobrano {len(data)} wierszy.")
            else:
                print(f"   -> PUSTE DANE (sprawdź uprawnienia dla {eod_symbol})")
        else:
            print(f"   -> BŁĄD HTTP: {response.status_code}")

    except Exception as e:
        print(f"   -> Błąd krytyczny: {e}")

    # Mała przerwa, żeby nie zablokowali klucza
    time.sleep(0.5)

# --- TWORZENIE TABELI ---
if all_data:
    df = pd.DataFrame(all_data)

    # Przesuńmy kolumnę symbol na początek, żeby było czytelnie
    cols = ['original_symbol', 'date', 'open', 'high', 'low', 'close', 'adjusted_close', 'volume']
    # Wybieramy tylko te kolumny, które istnieją w danych
    cols = [c for c in cols if c in df.columns]

    df_final = df[cols]

    print("\n--- SUKCES! GOTOWA TABELA ---")
    print(df_final.head(10))
    print(f"\nŁącznie pobrano: {len(df_final)} wierszy.")

    # Opcjonalnie: Zapisz do CSV
    df_final.to_csv("wszystkie_akcje_2025.csv", index=False)
else:
    print("\nNiestety nie udało się pobrać żadnych danych.")

--- Rozpoczynam pobieranie dla 8 symboli ---
Pobieranie: AAPL (AAPL.US)...
   -> OK. Pobrano 244 wierszy.
Pobieranie: AMZN (AMZN.US)...
   -> OK. Pobrano 244 wierszy.
Pobieranie: NVDA (NVDA.US)...
   -> OK. Pobrano 244 wierszy.
Pobieranie: INTC (INTC.US)...
   -> OK. Pobrano 244 wierszy.
Pobieranie: AMD (AMD.US)...
   -> OK. Pobrano 244 wierszy.
Pobieranie: BRK.B (BRK-B.US)...
   -> OK. Pobrano 244 wierszy.
Pobieranie: BINANCE:BTCUSDT (BTC-USD.CC)...
   -> OK. Pobrano 355 wierszy.
Pobieranie: BINANCE:ETHUSDT (ETH-USD.CC)...
   -> OK. Pobrano 355 wierszy.

--- SUKCES! GOTOWA TABELA ---
  original_symbol        date    open    high     low   close  adjusted_close  \
0            AAPL  2025-01-10  240.01  240.16  233.00  236.85        235.7836   
1            AAPL  2025-01-13  233.53  234.67  229.72  234.40        233.3446   
2            AAPL  2025-01-14  234.75  236.12  232.47  233.28        232.2297   
3            AAPL  2025-01-15  234.64  238.96  234.43  237.87        236.7990   
4  

In [4]:
len(df)

2174

In [44]:
# url = "https://eodhd.com/api/fundamentals/AAPL.US?api_token=demo"
#
# response = requests.get(url)
#
# data = response.json()

In [45]:
# print(data)

{'General': {'Code': 'AAPL', 'Type': 'Common Stock', 'Name': 'Apple Inc', 'Exchange': 'NASDAQ', 'CurrencyCode': 'USD', 'CurrencyName': 'US Dollar', 'CurrencySymbol': '$', 'CountryName': 'USA', 'CountryISO': 'US', 'OpenFigi': 'BBG000B9XRY4', 'ISIN': 'US0378331005', 'LEI': 'HWUPKR0MPOU8FGXBT394', 'PrimaryTicker': 'AAPL.US', 'CUSIP': '037833100', 'CIK': '0000320193', 'EmployerIdNumber': '94-2404110', 'FiscalYearEnd': 'September', 'IPODate': '1980-12-12', 'InternationalDomestic': 'International/Domestic', 'Sector': 'Technology', 'Industry': 'Consumer Electronics', 'GicSector': 'Information Technology', 'GicGroup': 'Technology Hardware & Equipment', 'GicIndustry': 'Technology Hardware, Storage & Peripherals', 'GicSubIndustry': 'Technology Hardware, Storage & Peripherals', 'HomeCategory': 'Domestic', 'IsDelisted': False, 'Description': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a li

In [46]:
# import json
#
# # pretty print json
# print(json.dumps(data, indent=4))

{
    "General": {
        "Code": "AAPL",
        "Type": "Common Stock",
        "Name": "Apple Inc",
        "Exchange": "NASDAQ",
        "CurrencyCode": "USD",
        "CurrencyName": "US Dollar",
        "CurrencySymbol": "$",
        "CountryName": "USA",
        "CountryISO": "US",
        "OpenFigi": "BBG000B9XRY4",
        "ISIN": "US0378331005",
        "LEI": "HWUPKR0MPOU8FGXBT394",
        "PrimaryTicker": "AAPL.US",
        "CUSIP": "037833100",
        "CIK": "0000320193",
        "EmployerIdNumber": "94-2404110",
        "FiscalYearEnd": "September",
        "IPODate": "1980-12-12",
        "InternationalDomestic": "International/Domestic",
        "Sector": "Technology",
        "Industry": "Consumer Electronics",
        "GicSector": "Information Technology",
        "GicGroup": "Technology Hardware & Equipment",
        "GicIndustry": "Technology Hardware, Storage & Peripherals",
        "GicSubIndustry": "Technology Hardware, Storage & Peripherals",
        "HomeCat

In [47]:
# # change json to dataframe
# fundamentals_df = pd.json_normalize(data)
# print(fundamentals_df.T.head(50))

                                                                               0
General.Code                                                                AAPL
General.Type                                                        Common Stock
General.Name                                                           Apple Inc
General.Exchange                                                          NASDAQ
General.CurrencyCode                                                         USD
General.CurrencyName                                                   US Dollar
General.CurrencySymbol                                                         $
General.CountryName                                                          USA
General.CountryISO                                                            US
General.OpenFigi                                                    BBG000B9XRY4
General.ISIN                                                        US0378331005
General.LEI                 

In [48]:
# import requests
#
# url = f'https://eodhd.com/api/ticks/?s=AAPL&from=1694455200&to=1694541600&limit=1&api_token=demo&fmt=json'
# data = requests.get(url).json()
#
# print(data)

{'mkt': ['V'], 'price': [179.25], 'seq': [376150475], 'shares': [100], 'sl': ['@   '], 'sub_mkt': [''], 'ts': [1694455200017]}


In [49]:
# import requests
#
# url = f'https://eodhd.com/api/fundamentals/BTC-USD.CC?api_token=demo&fmt=json'
# data = requests.get(url).json()
#
# print(data)

{'General': {'Name': 'Bitcoin', 'Type': 'Crypto', 'Category': 'coin', 'WebURL': 'https://bitcoin.org/', 'Description': 'Bitcoin is a cryptocurrency and worldwide payment system. It is the first decentralized digital currency, as the system works without a central bank or single administrator.'}, 'Tech': {'Developers': {'0': 'Satoshi Nakamoto - Founder', '1': 'Wladimir J. van der Laan - Blockchain Developer', '2': 'Jonas Schnelli - Blockchain Developer', '3': 'Marco Falke - Blockchain Developer'}}, 'Resources': {'Links': {'reddit': {'0': 'https://www.reddit.com/r/bitcoin'}, 'website': {'0': 'https://bitcoin.org/'}, 'youtube': {'0': 'https://www.youtube.com/watch?v=Gc2en3nHxA4&'}, 'explorer': {'0': 'http://blockchain.com/explorer', '1': 'https://blockstream.info/', '2': 'https://blockchair.com/bitcoin', '3': 'https://live.blockcypher.com/btc/', '4': 'https://btc.cryptoid.info/btc/'}, 'facebook': {'0': 'https://www.facebook.com/bitcoins/'}, 'source_code': {'0': 'https://github.com/bitcoin

In [51]:
# import requests
#
# url = f'https://eodhd.com/api/fundamentals/AAPL.US?filter=General::Code,General,Earnings&api_token=demo&fmt=json'
# data = requests.get(url).json()
#
# print(data)

{'General::Code': 'AAPL', 'General': {'Code': 'AAPL', 'Type': 'Common Stock', 'Name': 'Apple Inc', 'Exchange': 'NASDAQ', 'CurrencyCode': 'USD', 'CurrencyName': 'US Dollar', 'CurrencySymbol': '$', 'CountryName': 'USA', 'CountryISO': 'US', 'OpenFigi': 'BBG000B9XRY4', 'ISIN': 'US0378331005', 'LEI': 'HWUPKR0MPOU8FGXBT394', 'PrimaryTicker': 'AAPL.US', 'CUSIP': '037833100', 'CIK': '0000320193', 'EmployerIdNumber': '94-2404110', 'FiscalYearEnd': 'September', 'IPODate': '1980-12-12', 'InternationalDomestic': 'International/Domestic', 'Sector': 'Technology', 'Industry': 'Consumer Electronics', 'GicSector': 'Information Technology', 'GicGroup': 'Technology Hardware & Equipment', 'GicIndustry': 'Technology Hardware, Storage & Peripherals', 'GicSubIndustry': 'Technology Hardware, Storage & Peripherals', 'HomeCategory': 'Domestic', 'IsDelisted': False, 'Description': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The co

In [52]:
# symbol = "KRU.WAR"
#
# # Budowanie URL
# url = f'https://eodhd.com/api/eod/{symbol}?api_token={api_token}&fmt=json'
#
# # Wysłanie zapytania
# response = requests.get(url)
#
# # Sprawdzenie czy wszystko ok (kod 200)
# if response.status_code == 200:
#     data = response.json()
#
#     # Opcjonalnie: Konwersja do ładnej tabeli DataFrame (jeśli używasz pandas)
#     df_kruk = pd.DataFrame(data)
#     print(df_kruk.head()) # Wyświetl 5 pierwszych wierszy
# else:
#     print("Błąd pobierania danych:", response.status_code)

         date   open   high    low  close  adjusted_close  volume
0  2024-11-18  423.2  427.4  418.2  420.8        402.1851   18512
1  2024-11-19  424.0  424.8  405.8  408.2        390.1425   37439
2  2024-11-20  413.2  417.4  407.0  411.2        393.0098   17505
3  2024-11-21  411.2  420.6  406.4  418.6        400.0824   25502
4  2024-11-22  418.8  422.0  411.4  420.0        401.4205   13149


In [53]:
# # Budowanie URL
# url = f'https://eodhd.com/api/eod/{symbol}?period=w&api_token={api_token}&fmt=json'
#
# # Wysłanie zapytania
# response = requests.get(url)
#
# # Sprawdzenie czy wszystko ok (kod 200)
# if response.status_code == 200:
#     data = response.json()
#
#     # Opcjonalnie: Konwersja do ładnej tabeli DataFrame (jeśli używasz pandas)
#     df_kruk = pd.DataFrame(data)
#     print(df_kruk.head()) # Wyświetl 5 pierwszych wierszy
# else:
#     print("Błąd pobierania danych:", response.status_code)

         date   open   high    low  close  adjusted_close  volume
0  2024-11-18  423.2  427.4  405.8  420.0        401.4205  112107
1  2024-11-25  423.0  427.2  417.0  423.0        404.2878   56758
2  2024-12-02  425.0  446.4  423.0  438.0        418.6242  102496
3  2024-12-09  444.0  446.8  428.2  434.6        415.3746   96283
4  2024-12-16  436.6  439.8  411.0  419.4        400.8470  156767
